# Full Pipeline Demo

This notebook loads the trained **Stage 1** (Mask R-CNN tooth segmentation) and **Stage 2** (anomaly classifier) checkpoints, runs the full pipeline on panoramic dental X-rays, and visualizes each tooth colored by its anomaly status.

## Pipeline Flow
```
Panoramic X-ray
     ↓
Stage 1: Mask R-CNN  →  tooth boxes + masks + tooth IDs
     ↓
Stage 2: ResNet-18   →  NORMAL / ANOMALY per tooth
     ↓
Visualization + Clinical Report
```

**Green** = Normal tooth | **Red** = Anomaly tooth

In [ ]:
# ================== CELL 1: IMPORTS ==================
import os
import json
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import nms

print(f'PyTorch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print('✓ Imports complete')

In [ ]:
# ================== CELL 2: CONFIG & PATHS ==================
CONFIG = {
    'stage1_conf_threshold': 0.6,
    'stage1_nms_iou': 0.3,
    'stage2_threshold': 0.5,
    'stage2_img_size': 224,
    'crop_padding': 8,
}

# Update these paths after attaching the Stage 1 and Stage 2 notebook outputs in Kaggle
STAGE1_CKPT = '/kaggle/input/stage1-output/maskrcnn_teeth_best.pth'
STAGE2_CKPT = '/kaggle/input/stage2-output/stage2_anomaly_best.pth'
IMG_DIR     = '/kaggle/input/datasets/humansintheloop/teeth-segmentation-on-dental-x-ray-images/Teeth Segmentation PNG/d2/img'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Stage 1 ckpt:', STAGE1_CKPT)
print('Stage 2 ckpt:', STAGE2_CKPT)
print('Image dir   :', IMG_DIR)
print('Device      :', device)
print('✓ Config complete')

In [ ]:
# ================== CELL 3: SCAN FOR .PTH FILES ==================
# Helper to find attached checkpoint files in /kaggle/input
def find_pth_files(root='/kaggle/input'):
    found = []
    for r, d, files in os.walk(root):
        for f in files:
            if f.endswith('.pth'):
                found.append(os.path.join(r, f))
    return found

pth_files = find_pth_files()
print('Found checkpoint files:')
for p in pth_files:
    print(' ', p)
if not pth_files:
    print('  No .pth files found — attach Stage 1 and Stage 2 notebook outputs first.')
print('✓ Scan complete')

In [ ]:
# ================== CELL 4: MODEL DEFINITIONS ==================

def get_model_instance_segmentation(num_classes: int):
    """Mask R-CNN with custom heads (same architecture as Stage 1 training)."""
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(
        weights=None, weights_backbone=None
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


def get_anomaly_classifier():
    """ResNet-18 binary classifier (same architecture as Stage 2 training)."""
    backbone = models.resnet18(weights=None)
    in_features = backbone.fc.in_features
    backbone.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 1)
    )
    return backbone


print('✓ Model builders ready')

In [ ]:
# ================== CELL 5: LOAD CHECKPOINTS ==================

# --- Stage 1 ---
stage1_model = get_model_instance_segmentation(num_classes=33)
ckpt1 = torch.load(STAGE1_CKPT, map_location=device)
if isinstance(ckpt1, dict) and 'model_state_dict' in ckpt1:
    stage1_model.load_state_dict(ckpt1['model_state_dict'])
    print(f"Loaded Stage 1 — best val_loss: {ckpt1.get('val_loss', 'N/A')}")
else:
    stage1_model.load_state_dict(ckpt1)
    print('Loaded Stage 1 from raw state_dict')
stage1_model.to(device)
stage1_model.eval()
print('✓ Stage 1 loaded')

# --- Stage 2 ---
stage2_model = get_anomaly_classifier()
ckpt2 = torch.load(STAGE2_CKPT, map_location=device)
if isinstance(ckpt2, dict) and 'model_state_dict' in ckpt2:
    stage2_model.load_state_dict(ckpt2['model_state_dict'])
    print(f"Loaded Stage 2 — best val metric: {ckpt2.get('val_loss', 'N/A')}")
else:
    stage2_model.load_state_dict(ckpt2)
    print('Loaded Stage 2 from raw state_dict')
stage2_model.to(device)
stage2_model.eval()
print('✓ Stage 2 loaded')

In [ ]:
# ================== CELL 6: STAGE 1 INFERENCE HELPERS ==================

def apply_nms(predictions, iou_threshold=CONFIG['stage1_nms_iou']):
    if len(predictions['boxes']) == 0:
        return predictions
    keep = nms(
        boxes=predictions['boxes'],
        scores=predictions['scores'],
        iou_threshold=iou_threshold
    )
    return {k: v[keep] for k, v in predictions.items()}


def run_stage1_inference(
    model, image_tensor, device,
    confidence_threshold=CONFIG['stage1_conf_threshold']
):
    model.eval()
    with torch.no_grad():
        raw_pred = model(image_tensor.to(device).unsqueeze(0))[0]
    keep = raw_pred['scores'] >= confidence_threshold
    filtered = {k: v[keep] for k, v in raw_pred.items()}
    filtered = apply_nms(filtered)
    return {
        'boxes':  filtered['boxes'].cpu().numpy(),
        'labels': filtered['labels'].cpu().numpy(),
        'masks':  filtered['masks'].cpu().numpy(),
        'scores': filtered['scores'].cpu().numpy(),
    }


print('✓ Stage 1 inference helpers ready')

In [ ]:
# ================== CELL 7: STAGE 2 INFERENCE HELPERS ==================

_STAGE2_TFM = transforms.Compose([
    transforms.Resize((CONFIG['stage2_img_size'], CONFIG['stage2_img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def crop_tooth(image_np, box, padding=CONFIG['crop_padding']):
    h, w = image_np.shape[:2]
    x1 = max(0, int(box[0]) - padding)
    y1 = max(0, int(box[1]) - padding)
    x2 = min(w, int(box[2]) + padding)
    y2 = min(h, int(box[3]) + padding)
    return Image.fromarray(image_np[y1:y2, x1:x2])


def classify_tooth(crop_pil, model, device, threshold=CONFIG['stage2_threshold']):
    x = _STAGE2_TFM(crop_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()
    return {
        'label': 'ANOMALY' if prob >= threshold else 'NORMAL',
        'probability': round(prob, 4)
    }


def run_full_pipeline(image_np, image_tensor):
    """Run Stage 1 + Stage 2 on a single image."""
    stage1 = run_stage1_inference(stage1_model, image_tensor, device)
    results = []
    for box, tooth_id, seg_score in zip(stage1['boxes'], stage1['labels'], stage1['scores']):
        crop = crop_tooth(image_np, box)
        cls  = classify_tooth(crop, stage2_model, device)
        results.append({
            'tooth_id':          int(tooth_id),
            'box':               box.tolist(),
            'segmentation_score': round(float(seg_score), 4),
            'anomaly_label':     cls['label'],
            'anomaly_probability': cls['probability'],
        })
    return stage1, results


print('✓ Stage 2 inference helpers ready')

In [ ]:
# ================== CELL 8: VISUALIZATION ==================

NORMAL_COLOR  = (40,  200, 80)
ANOMALY_COLOR = (220, 40,  40)


def visualize_pipeline(img_path):
    img = Image.open(img_path).convert('RGB')
    image_np     = np.array(img, dtype=np.uint8).copy()
    image_tensor = torch.from_numpy(image_np.copy()).permute(2, 0, 1).float() / 255.0

    stage1, results = run_full_pipeline(image_np, image_tensor)
    n_anom = sum(1 for r in results if r['anomaly_label'] == 'ANOMALY')
    print(f'  {len(results)} teeth detected | {n_anom} flagged as anomaly')

    overlay = image_np.copy()
    for r, mask_raw in zip(results, stage1['masks']):
        mask_bin = (mask_raw[0] > 0.5).astype(np.uint8)
        color    = ANOMALY_COLOR if r['anomaly_label'] == 'ANOMALY' else NORMAL_COLOR
        for c in range(3):
            ch = overlay[:, :, c].astype(np.float32)
            ch[mask_bin == 1] = 0.35 * color[c] + 0.65 * ch[mask_bin == 1]
            overlay[:, :, c] = ch.astype(np.uint8)
        contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, color, 3)
        x1, y1, x2, y2 = [int(v) for v in r['box']]
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        label_str = f"T{r['tooth_id']}"
        cv2.putText(overlay, label_str, (cx - 20, cy + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 4)
        cv2.putText(overlay, label_str, (cx - 20, cy + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        if r['anomaly_label'] == 'ANOMALY':
            cv2.putText(overlay, f"{r['anomaly_probability']:.2f}",
                        (cx - 18, cy + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,220,0), 2)

    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    fig.suptitle(
        f'Full Pipeline — {os.path.basename(img_path)}  '
        f'({n_anom}/{len(results)} anomalies)',
        fontsize=14, fontweight='bold'
    )
    axes[0].imshow(image_np);  axes[0].set_title('Original X-ray', fontsize=13); axes[0].axis('off')
    axes[1].imshow(overlay);   axes[1].set_title('Stage 1 + Stage 2 Output', fontsize=13); axes[1].axis('off')
    axes[1].legend(handles=[
        mpatches.Patch(color=(40/255, 200/255, 80/255), label='Normal'),
        mpatches.Patch(color=(220/255, 40/255, 40/255), label='Anomaly'),
    ], loc='lower right', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"{'Tooth':>6}  {'Label':<10}  {'P(anomaly)':>12}  {'SegScore':>10}")
    print('-' * 50)
    for r in sorted(results, key=lambda x: x['tooth_id']):
        print(f"T{r['tooth_id']:>2d}    {r['anomaly_label']:<10}  "
              f"{r['anomaly_probability']:>12.4f}  {r['segmentation_score']:>10.4f}")


print('✓ Visualization function ready')

In [ ]:
# ================== CELL 9: RUN DEMO ==================

all_img_files = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.lower().endswith(('.jpg', '.png', '.jpeg'))
])
print(f'Total images available: {len(all_img_files)}')

NUM_SAMPLES = min(3, len(all_img_files))
for i in range(NUM_SAMPLES):
    print('\n' + '=' * 60)
    print(f'SAMPLE {i+1}/{NUM_SAMPLES}: {all_img_files[i]}')
    print('=' * 60)
    visualize_pipeline(os.path.join(IMG_DIR, all_img_files[i]))

print('\n✓ FULL PIPELINE DEMO COMPLETE')